In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import itertools
import shap
import lime
shap.initjs()

from sklearn.neural_network import MLPRegressor
from sklearn.model_selection import RandomizedSearchCV

# imports from "common" folder
import common.common_functions as cf
from common.data_processing import process_data

### 1. Data Processing

In [ ]:
# Data processing from data_processing.py
train, test, y_train, y_test, X_train_scaled, X_test_scaled = process_data()

### 2. Multilayer Perceptron (MLP) Model with Hyperparameter Tunining using RandomizedSearchCV

In [ ]:
# Create the parameter grid

# Exploring diiferent combinations of upto three hidden layers, with different number of neurons per layer.
layers = [2, 3]
neurones = [50, 100, 150, 200]

comb_array = []

for layer_count in layers:
    comb = list(itertools.combinations(neurones, layer_count))
    comb_array.extend(comb)  

print(comb_array)

param_grid = {'hidden_layer_sizes': comb_array,
              'activation': ['tanh','relu', 'identity', 'logistic'],
              'solver': ['sgd', 'adam'],        
              'learning_rate': ['constant', 'invscaling', 'adaptive'],
              'alpha': [0.0001, 0.05, 0.001, 0.01, 0.1],
              'max_iter': [200, 500, 1000,1500], 
              'batch_size': [10, 30, 50]
              }
print(param_grid)

In [ ]:
# Hyperparameter tunining 
MLP_model = MLPRegressor(random_state= 42)
MLP_RandomGrid = RandomizedSearchCV(estimator =  MLP_model, param_distributions = param_grid, cv = 5, verbose=2, n_jobs = -1, random_state= 42)
MLP_RandomGrid.fit(X_train_scaled, y_train)
print(MLP_RandomGrid.best_params_)

Predictions_train =  MLP_RandomGrid.predict(X_train_scaled)
Predictions_test =  MLP_RandomGrid.predict(X_test_scaled)

In [ ]:
# Save the predictions from the model
train_predicted_data = pd.DataFrame({
    'Actual': y_train,  
    'Predicted': Predictions_train  
})

train_predicted_data.to_csv('../data/MLP_Predictions_training_I.csv', index=True)

test_predicted_data = pd.DataFrame({
    'Actual': y_test,  
    'Predicted': Predictions_test 
})

test_predicted_data.to_csv('../data/MLP_Predictions_test_I.csv', index=True)

In [ ]:
# Plot predicted vs observed MHCC plot from common_funtions.py
cf.plot_prd_vs_obs(y_train, Predictions_train, y_test, Predictions_test)

In [ ]:
# Model performance metrics using performance_metrics.py
cf.calculate_performance_metrics(y_train, Predictions_train, y_test, Predictions_test)

### 3. SHAP for Train Data

In [ ]:
# Compute SHAP values for the training data
np.random.seed(42)
explainer = shap.KernelExplainer(MLP_RandomGrid.best_estimator_.predict, shap.kmeans(X_train_scaled,100))
shap_values = explainer(X_train_scaled)

In [ ]:
# Get feature_names form common_functions.py
feature_names = cf.get_feature_names(X_train_scaled.columns)

In [ ]:
# Save the SHAP dataframe
shap_RF_df = pd.DataFrame(shap_values.values, columns= feature_names)
shap_RF_df.to_csv ("../data/SHAP_MLP_df.csv", index = True)

In [ ]:
# Create SHAP explanation object for visualisation
shap_values_with_names = shap.Explanation(
    values=shap_values.values,
    base_values=shap_values.base_values,
    data=shap_values.data,
    feature_names= feature_names
)

In [ ]:
# Get the projected SHAP vector removing particular dimensions that are not of interest.
remove_features = {"longitude", "latitude", "year"}
keep_idx = [
    i for i, name in enumerate(shap_values_with_names.feature_names)
    if name not in remove_features
]

# Create projected SHAP explanation object for visualisation
shap_values_filtered = shap.Explanation(
    values=shap_values_with_names.values[:, keep_idx],
    base_values=shap_values_with_names.base_values,
    data=shap_values_with_names.data[:, keep_idx] if shap_values_with_names.data is not None else None,
    feature_names=[feature_names[i] for i in keep_idx]
)

#### 3.1 Waterfall plot for high EDM common outliers of EDM (E1)

In [ ]:
# Waterfall plot using the full SHAP vector for common outliers of EDM (E1).

# 1-based indices correspond to E1 cases. These common outliers were found from EDM_SHAP.ipynb.
outlier_indices = [294, 318, 527, 539, 900, 1025, 1255, 1419, 1910, 2047, 2158, 2177]

for idx_1based in outlier_indices:
    idx = idx_1based - 1 # Converting to 0-based index

    site = train.iloc[idx]['site_name']
    year = train.iloc[idx]['year']

    print(f"1-based-index {idx_1based}, site={site}, year={year}")

    shap.plots.waterfall(shap_values_with_names[idx], max_display = 12, show=False)
    plt.gcf().suptitle(f"MLP - {site}, Year {year}", fontsize=14)
    plt.show() 

In [ ]:
# Waterfall plot using the projected SHAP vector for the representative E1 point in the paper.

outlier_index = 1419 # 1-based index correspond to E1
idx = outlier_index - 1 # Converting to 0-based index

row_index = train.index[idx]
site = train.iloc[idx]['site_name']
year = train.iloc[idx]['year']

print(f"1-based-index {outlier_index}, site={site}, year={year}")

shap.plots.waterfall(shap_values_filtered[idx], max_display = 4, show=False)
plt.gcf().suptitle(f"MLP - {site}, Year {year}", fontsize=14)
plt.show()

#### 3.2 Waterfall plot for least EDM points (E3)

In [ ]:
# Waterfall plot using the full SHAP vector for least EDM points (E3).

# 1-based indexes correspond to the least SHAP discrepancy points (E3). These indices were found from EDM_SHAP.ipynb.
least_SHAP_dis_indices = [1245, 2536, 200, 2304, 663, 1116]

for idx_1based in least_SHAP_dis_indices:
    idx = idx_1based - 1 # Converting to 0-based index

    site = train.iloc[idx]['site_name']
    year = train.iloc[idx]['year']

    print(f"1-based-index {idx_1based}, site={site}, year={year}")

    shap.plots.waterfall(shap_values_with_names[idx], max_display = 12, show=False)
    plt.gcf().suptitle(f"MLP - {site}, Year {year}", fontsize=14)
    plt.show() 

In [ ]:
# Waterfall plot using the projected SHAP vector for the representative E3 point in the paper.

least_SHAP_dis_index = 200 # 1-based index correspond to E3
idx = least_SHAP_dis_index  - 1 # Converting to 0-based index

site = train.iloc[idx]['site_name']
year = train.iloc[idx]['year']

print(f"1-based-index {least_SHAP_dis_index }, site={site}, year={year}")

shap.plots.waterfall(shap_values_filtered[idx], max_display = 4, show=False)
plt.gcf().suptitle(f"MLP - {site}, Year {year}", fontsize=14)
plt.show()

#### 3.3 Waterfall plot for clear cyclone events with low EDM value (E2)

In [ ]:
# Waterfall plot using the full SHAP vector for clear cyclone events with low EDM value (E2).

# Define site_name and year for E2 to plot. 
clear_cyc_events = [
    ("Reef18 Site 1", 6),
    ("Reef25 Site 1", 6),
    ("Reef30 Site 1", 6),
    ("Reef30 Site 1", 12),
    ("Reef36 Site 1", 6),
    ("Reef37 Site 2", 12),
    ("Reef39 Site 1", 12),
    ("Reef41 Site 1", 12),
    ("Reef44 Site 1", 6),  
    ("Reef45 Site 1", 6),
    ("Reef46 Site 3", 6),
    ("Reef46 Site 2", 6)
]

train_reindexed = train.reset_index(drop=True)

for site, yr in clear_cyc_events:
    mask = ((train_reindexed["site_name"] == site) & (train_reindexed["year"] == yr))
    idx_list = train_reindexed.index[mask] 

    print(f"{site}, Year {yr}")

    # Plot waterfall plot
    for idx in idx_list:
        row_label = (f"{site}, Year {yr}")
        shap.plots.waterfall(shap_values_with_names[idx],  max_display = 12, show=False)
        plt.gcf().suptitle(f"MLP - {row_label}", fontsize=14)
        plt.tight_layout()
        plt.show()

In [ ]:
# Waterfall plot using the projected SHAP vector for the representative E2 point in the paper.

# Define site_name and year for E2 
clear_cyc_event_site = "Reef25 Site 1"
clear_cyc_event_year = 6

train_reindexed = train.reset_index(drop=True)

mask = ((train_reindexed["site_name"] == clear_cyc_event_site) & (train_reindexed["year"] == clear_cyc_event_year))
idx = train_reindexed.index[mask][0]  

print(f"Row for ({clear_cyc_event_site}, {clear_cyc_event_year}): {idx}")

# Plot a waterfall 
row_label = f"{clear_cyc_event_site}, Year {clear_cyc_event_year}"
shap.plots.waterfall(shap_values_filtered[idx], max_display = 4, show=False)
plt.gcf().suptitle(f"MLP - {row_label}", fontsize=14)
plt.tight_layout()
plt.show()


#### 3.4 SHAP summary plot (Beeswarm plot)

In [ ]:
# Plot the beeswarm plot
# In these beeswarm plot, the features are ordered alphabetically, for direct comparison across different models.
sorted_features = sorted(feature_names)  
col2num = {col: i for i, col in enumerate(feature_names)} 
order = list(map(col2num.get, sorted_features))  

shap.plots.beeswarm(shap_values_with_names, max_display=12, order=order, show=False)
plt.gcf().suptitle("MLP - Training", fontsize=16)
plt.show()

#### 3.5 Mean SHAP

In [ ]:
# Plot the mean SHAP 
shap.plots.bar(shap_values_with_names, show=False, max_display=6)
plt.gcf().suptitle("MLP - Training", fontsize=16)
plt.show()

### 4. SHAP for Test Data

In [ ]:
# Compute SHAP values for the test data
np.random.seed(42)
explainer = shap.KernelExplainer(MLP_RandomGrid.best_estimator_.predict, shap.kmeans(X_train_scaled,50))
shap_values = explainer(X_test_scaled)

In [ ]:
# Save the SHAP dataframe
shap_RF_df = pd.DataFrame(shap_values.values, columns= feature_names)
shap_RF_df.to_csv ("../data/SHAP_test_MLP_df.csv", index = True)


In [ ]:
# Create SHAP explanation object for visualisation
shap_values_with_names = shap.Explanation(
    values=shap_values.values,
    base_values=shap_values.base_values,
    data=shap_values.data,
    feature_names= feature_names
)

#### 4.1 SHAP summary plot (Beeswarm plot)

In [ ]:
# Plot the beeswarm plot
# In these beeswarm plot, the features are ordered alphabetically, for direct comparison across different models.
sorted_features = sorted(feature_names)  # Alphabetical order
col2num = {col: i for i, col in enumerate(feature_names)}  # Original indices
order = list(map(col2num.get, sorted_features))  # Map sorted features to their indices

shap.plots.beeswarm(shap_values_with_names, max_display=22, order=order, show=False)
plt.gcf().suptitle("MLP - Test", fontsize=16)
plt.show()

#### 4.2 Mean SHAP

In [ ]:
# Plot the mean SHAP 
shap.plots.bar(shap_values_with_names, show=False, max_display=6)
plt.gcf().suptitle("MLP - Test", fontsize=16)
plt.show()

### LIME

In [ ]:
# for E1, E3
# indices = [1419, 200]]
indices = [1718, 1384]
for idx_1based in indices:

    idx = idx_1based - 1

    site = train.iloc[idx]["site_name"]
    year = train.iloc[idx]["year"]

    print(f"\nSite: {site}, Year: {year}")

    exp = cf.get_lime_explanation(
        idx,
        X_train_scaled,
        feature_names,
        MLP_RandomGrid.best_estimator_
    )

    exp.show_in_notebook()

    fig = exp.as_pyplot_figure()
    ax = fig.axes[0]
    ax.set_title("MLP")
    plt.tight_layout()

In [ ]:
# For E2
# clear_cyc_event_site = "Reef25 Site 1"
# clear_cyc_event_year = 6

clear_cyc_event_site = "Reef30 Site 1"
clear_cyc_event_year = 6

train_reindexed = train.reset_index(drop=True)

mask = (
    (train_reindexed["site_name"] == clear_cyc_event_site) & 
    (train_reindexed["year"] == clear_cyc_event_year)
)
idx = train_reindexed.index[mask][0]

exp = cf.get_lime_explanation(
    idx,
    X_train_scaled,
    feature_names,
    MLP_RandomGrid.best_estimator_
)

exp.show_in_notebook()
# exp.as_pyplot_figure()

fig = exp.as_pyplot_figure()
ax = fig.axes[0]
ax.set_title("MLP")
plt.tight_layout()

In [ ]:
def return_weights(exp):

    """ Get weights from LIME explanation object """
    exp_list = exp.as_map()[1]
    exp_list = sorted(exp_list, key=lambda x: x[0])
    exp_weight = [x[1] for x in exp_list]
    return exp_weight

In [ ]:
lime_weights = []

for idx in range(len(X_train_scaled)):

    exp = cf.get_lime_explanation(
        idx,
        X_train_scaled,
        feature_names,
        MLP_RandomGrid.best_estimator_
    )

    lime_weights.append(return_weights(exp))

lime_MLP_df = pd.DataFrame(
    lime_weights,
    columns=feature_names,
    index=X_train_scaled.index
)

In [ ]:
lime_MLP_df.to_csv("../data/LIME_dcf_MLP_df.csv")